# sequential llm call -> tool call -> llm call function

In [1]:
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
openai_client = OpenAI()

In [3]:
def get_ticket_prices(city: str) -> str:
    FLIGHT_PRICES = {
        "london": "$550",
        "paris": "$600",
        "new york": "$450",
        "tokyo": "$950",
        "berlin": "$500"
    }
    cleaned_city = city.lower().strip()
    
    print(f"\n[Tool Executing] Looking up flight prices for: '{cleaned_city}'")
    
    if cleaned_city in FLIGHT_PRICES:
        price = FLIGHT_PRICES[cleaned_city]
        return f"The ticket for {city.title()} is {price}."
    
    return f"Tickets for {city.title()} are currently not available."

In [4]:
def get_weather(city: str) -> str:
    WEATHER_DATA = {
        "london": "12°C, Rainy",
        "paris": "15°C, Partly Cloudy",
        "new york": "18°C, Sunny",
        "tokyo": "20°C, Clear",
        "berlin": "10°C, Foggy"
    }
    cleaned_city = city.lower().strip()
    
    print(f"\n[Tool Executing] Looking up weather for: '{cleaned_city}'")
    
    if cleaned_city in WEATHER_DATA:
        condition = WEATHER_DATA[cleaned_city]
        return f"The weather in {city.title()} is currently {condition}."
    
    return f"Weather data for {city.title()} is currently not available."


In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_prices",
            "description": "Retrieves the flight ticket price for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to look up ticket prices for (e.g., 'Paris', 'London')."
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieves the current weather condition and temperature for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to look up the weather for (e.g., 'Paris', 'New York')."
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    }
]


In [6]:
TOOLS_DIRECTORY = {
    "get_ticket_prices": get_ticket_prices,
    "get_weather": get_weather
}

In [7]:
def chat(user_prompt):
    system_message = {
        "role": "system",
        "content": "You are helpful airline assistant"
    }
    user_message = {
        "role": "user",
        "content": user_prompt
    }
    messages = [system_message] + [user_message]
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )
    messages.append(response.choices[0].message)
    tool_calls = response.choices[0].message.tool_calls
    if tool_calls:
        for tool_call in tool_calls:
            tool_function = TOOLS_DIRECTORY.get(tool_call.function.name, None)
            tool_arguments = json.loads(tool_call.function.arguments)
            if tool_function:
                result = tool_function(**tool_arguments)
                messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": str(result)
                    })
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )
    return response.choices[0].message.content

# testing

### 1. single variable for single tool function

In [8]:
print(chat("What are the ticket prices of london?"))


[Tool Executing] Looking up flight prices for: 'london'
The ticket price for London is $550.


### 2. multiple variable for single tool function

In [9]:
print(chat("What are the ticket prices of london, new york, paris?"))


[Tool Executing] Looking up flight prices for: 'london'

[Tool Executing] Looking up flight prices for: 'new york'

[Tool Executing] Looking up flight prices for: 'paris'
Here are the ticket prices for the cities you requested:

- London: $550
- New York: $450
- Paris: $600


### 3. single variables for multiple tool functions

In [10]:
print(chat("What is the weather and ticket price of london?"))


[Tool Executing] Looking up weather for: 'london'

[Tool Executing] Looking up flight prices for: 'london'
The current weather in London is 12°C and rainy. The ticket price for London is $550.


### 4. multiple variable for multiple tool functions

In [11]:
print(chat("What is the weather and ticket price of london, new york, paris?"))


[Tool Executing] Looking up weather for: 'london'

[Tool Executing] Looking up weather for: 'new york'

[Tool Executing] Looking up weather for: 'paris'

[Tool Executing] Looking up flight prices for: 'london'

[Tool Executing] Looking up flight prices for: 'new york'

[Tool Executing] Looking up flight prices for: 'paris'
Here are the weather conditions and ticket prices for London, New York, and Paris:

### Weather:
- **London**: 12°C, Rainy
- **New York**: 18°C, Sunny
- **Paris**: 15°C, Partly Cloudy

### Ticket Prices:
- **London**: $550
- **New York**: $450
- **Paris**: $600


# tricky part

### conditional questions, if tool(a) is this then what is tool(b)

In [12]:
print(chat("is the ticket price of paris is above $500?"))


[Tool Executing] Looking up flight prices for: 'paris'
Yes, the ticket price for Paris is $600, which is above $500.


In [13]:
print(chat("if ticket price of paris is above $500, give me weather info and ticket price for new york"))


[Tool Executing] Looking up flight prices for: 'paris'
None


# intro to multistep tool calling

-> we need a feedback loop from the llm again to decide if we want to use tool call again.

In [14]:
def chat(user_prompt, max_tool_calls=3):
    system_message = {
        "role": "system",
        "content": "You are helpful airline assistant"
    }
    user_message = {
        "role": "user",
        "content": user_prompt
    }
    messages = [system_message] + [user_message]

    tool_cycle = 0
    while tool_cycle < max_tool_calls:

        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )
        
        messages.append(response.choices[0].message)
        
        tool_calls = response.choices[0].message.tool_calls
        
        if tool_calls:
            tool_cycle += 1
            print(f"---in {tool_cycle}th tool execution cycle---")
            
            for tool_call in tool_calls:
                tool_function = TOOLS_DIRECTORY.get(tool_call.function.name, None)
                tool_arguments = json.loads(tool_call.function.arguments)
                if tool_function:
                    result = tool_function(**tool_arguments)
                    messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": tool_call.function.name,
                            "content": str(result)
                        })
        else:
            break

    else:
        print("Maximum tool calling execution limit reached!")

    # check if the last call was llm call or tool call
    # if it was tool call then summarise it with llm call
    last_msg = messages[-1]
    last_role = last_msg.get('role') if isinstance(last_msg, dict) else getattr(last_msg, 'role', '')

    if last_role == 'tool':
        # if last call was of tool and we have exhausted number of tool calls,
        # we say to the system that enough with the multi step requests!
        messages.append(
            {
                "role": "system",
                "content": "Maximum tool calling execution limit has been reached." 
                "Please respond to user with whatever content you have available."
                "Don't look for any more tools calls."
            }
        )
        # final summariser llm call
        final_turn = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        messages.append(final_turn.choices[0].message)

    print("---final response---")
    last_msg = messages[-1]
    role = last_msg.get('role') if isinstance(last_msg, dict) else getattr(last_msg, 'role', '')
    content = last_msg.get('content') if isinstance(last_msg, dict) else getattr(last_msg, 'content', '')
    
    if role == "assistant" and content:
        return content
            
    return "No response generated."

In [15]:
chat("what is weather like in london")

---in 1th tool execution cycle---

[Tool Executing] Looking up weather for: 'london'
---final response---


'The weather in London is currently 12°C and rainy.'

In [16]:
chat("what is ticket price for london and new york")

---in 1th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'london'

[Tool Executing] Looking up flight prices for: 'new york'
---final response---


'The ticket price for London is $550, and for New York, it is $450.'

In [17]:
chat("what is ticket price for london? "
     "if its ticket price is above $500 then check ticket price for new york and let me know.")

---in 1th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'london'
---in 2th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'new york'
---final response---


'The ticket price for London is $550. Since this is above $500, I checked the ticket price for New York, which is $450.'

In [18]:
chat("what is ticket price for london? "
    "if its ticket price is above $500 then check ticket price for new york and let me know."
    "if the ticket price for new york is below 500 and weather is not rainy then lets go ahead and book the ticket"
    )

---in 1th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'london'
---in 2th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'new york'
---in 3th tool execution cycle---

[Tool Executing] Looking up weather for: 'new york'
Maximum tool calling execution limit reached!
---final response---


"The ticket price for London is $550, which is above $500. The ticket price for New York is $450, which is below $500. The current weather in New York is sunny, so it's not rainy. \n\nLet's go ahead and book the ticket for New York!"

In [19]:
chat("what is ticket price for london? "
    "if its ticket price is above $500 then check ticket price for new york and let me know."
    "if the ticket price for new york is below 500 then check the weather for new york."
     "if weather of new york is sunny then check weather for berlin"
    )

---in 1th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'london'
---in 2th tool execution cycle---

[Tool Executing] Looking up flight prices for: 'new york'
---in 3th tool execution cycle---

[Tool Executing] Looking up weather for: 'new york'
Maximum tool calling execution limit reached!
---final response---


'The ticket price for London is $550, which is above $500. The ticket price for New York is $450, which is below $500. The weather in New York is currently sunny, with a temperature of 18°C.\n\nUnfortunately, I cannot check the weather for Berlin due to execution limits on tool calls, but I can assist with other inquiries you may have!'

In [20]:
chat("Hi How are you")

---final response---


"I'm here and ready to assist you! How can I help you today?"